In [1]:
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import glob
import pandas as pd

path = glob.glob("../data/online_retail*.xls*")[0]
print("Reading:", path)

sheets = pd.read_excel(path, sheet_name=None)
print(list(sheets.keys()))

df = pd.concat(sheets.values(), ignore_index=True)
df.to_csv("../data/online_retail_II.csv", index=False)
df.shape

Reading: ../data\online_retail_II.xlsx
['Year 2009-2010', 'Year 2010-2011']


(1067371, 8)

In [3]:
display(df.head())
df.info()
df.isna().sum()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [4]:
import pandas as pd

df = pd.read_csv(
    "../data/online_retail_II.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Invoice": str, "StockCode": str},
)
df.shape

(1067371, 8)

In [5]:
is_cancel = df["Invoice"].str.startswith("C")

print("Cancellation rows:", is_cancel.sum())
print("Negative quantity rows:", (df["Quantity"] < 0).sum())
print("Negative quantity, not cancellations:", ((df["Quantity"] < 0) & ~is_cancel).sum())
print("Price <= 0 rows:", (df["Price"] <= 0).sum())
print("Fully duplicated rows:", df.duplicated().sum())

non_product = df[~df["StockCode"].str.match(r"^\d{5}")]
non_product["StockCode"].value_counts().head(15)

Cancellation rows: 19494
Negative quantity rows: 22950
Negative quantity, not cancellations: 3457
Price <= 0 rows: 6207
Fully duplicated rows: 34335


StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
Name: count, dtype: int64

In [6]:
raw_rows = len(df)

df = df.drop_duplicates()
print("Rows after dropping exact duplicates:", len(df), "of", raw_rows)

is_cancel = df["Invoice"].str.startswith("C")
is_product = df["StockCode"].str.match(r"^\d{5}")

cancellations = df[is_cancel & is_product].copy()
cancellations["Revenue"] = cancellations["Quantity"] * cancellations["Price"]

sales = df[~is_cancel & is_product & (df["Quantity"] > 0) & (df["Price"] > 0)].copy()
sales["Revenue"] = sales["Quantity"] * sales["Price"]
sales["Customer ID"] = sales["Customer ID"].astype("Int64")

print("Clean sales rows:", len(sales))
print("Cancellation rows kept separately:", len(cancellations))
print("Sales rows without a customer ID:", sales["Customer ID"].isna().sum())

sales.to_csv("../data/sales_clean.csv", index=False)
cancellations.to_csv("../data/cancellations_clean.csv", index=False)

Rows after dropping exact duplicates: 1033036 of 1067371
Clean sales rows: 1003214
Cancellation rows kept separately: 17914
Sales rows without a customer ID: 226637


In [7]:
display(sales[["Quantity", "Price", "Revenue"]].describe())
display(sales.nlargest(5, "Quantity")[["Invoice", "Description", "Quantity", "Price", "Customer ID"]])

,Quantity,Price,Revenue
count,1.003214e+06,1.003214e+06,1.003214e+06
mean,1.115193e+01,3.339875e+00,1.957976e+01
std,1.287471e+02,4.754713e+00,1.999959e+02
min,1.000000e+00,3.000000e-02,6.000000e-02
25%,1.000000e+00,1.250000e+00,4.130000e+00
50%,4.000000e+00,2.100000e+00,1.008000e+01
75%,1.200000e+01,4.130000e+00,1.770000e+01
max,8.099500e+04,1.157150e+03,1.684696e+05


,Invoice,Description,Quantity,Price,Customer ID
1065882,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446
587080,541431,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346
90857,497946,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902
127166,501534,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902
127168,501534,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902


In [8]:
big = sales.nlargest(10, "Revenue")[
    ["Invoice", "InvoiceDate", "Description", "Quantity", "Price", "Revenue", "Customer ID"]
]
display(big)

for cust in [16446, 12346]:
    display(
        cancellations[cancellations["Customer ID"] == cust][
            ["Invoice", "InvoiceDate", "Description", "Quantity", "Price"]
        ]
    )

display(
    sales.nlargest(5, "Price")[
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
    ]
)

,Invoice,InvoiceDate,Description,Quantity,Price,Revenue,Customer ID
1065882,581483,2011-12-09 09:15:00,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,16446
587080,541431,2011-01-18 10:01:00,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,12346
748132,556444,2011-06-10 15:28:00,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.00,15098
432176,530715,2010-11-04 11:36:00,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.40,15838
228042,511465,2010-06-08 12:59:00,PINK PAPER PARASOL,3500,2.55,8925.00,18008
873786,567423,2011-09-20 11:05:00,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,7144.72,17450
578172,540815,2011-01-11 12:55:00,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,15749
686007,550461,2011-04-18 13:20:00,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,15749
461644,533027,2010-11-15 16:02:00,PAPER CHAIN KIT 50'S CHRISTMAS,835,6.95,5803.25,<NA>
379875,525968,2010-10-08 10:10:00,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,1.66,5179.20,15838


,Invoice,InvoiceDate,Description,Quantity,Price
1065883,C581484,2011-12-09 09:27:00,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08


,Invoice,InvoiceDate,Description,Quantity,Price
587085,C541433,2011-01-18 10:17:00,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04


,Invoice,StockCode,Description,Quantity,Price,Customer ID
192196,507637,84016,FLAG OF ST GEORGE CAR FLAG,1,1157.15,<NA>
136403,502451,84016,FLAG OF ST GEORGE CAR FLAG,1,867.79,<NA>
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,15098
748143,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,649.50,15098
180997,506571,84016,FLAG OF ST GEORGE CAR FLAG,1,408.40,<NA>


In [9]:
canc = cancellations.dropna(subset=["Customer ID"]).copy()
canc["Customer ID"] = canc["Customer ID"].astype("Int64")
canc["Quantity"] = -canc["Quantity"]
canc = canc[["Customer ID", "StockCode", "Quantity", "InvoiceDate", "Invoice"]].rename(
    columns={"InvoiceDate": "CancelDate", "Invoice": "CancelInvoice"}
)

m = sales.reset_index().merge(canc, on=["Customer ID", "StockCode", "Quantity"], how="inner")
m = m[(m["CancelDate"] >= m["InvoiceDate"]) & (m["CancelDate"] - m["InvoiceDate"] <= pd.Timedelta(days=1))]

reversed_idx = m["index"].unique()
print("Sales rows reversed within 1 day:", len(reversed_idx))
print("Revenue reversed:", round(sales.loc[reversed_idx, "Revenue"].sum(), 2))
display(m.nlargest(8, "Revenue")[["Invoice", "CancelInvoice", "Description", "Quantity", "Revenue"]])

display(sales[sales["StockCode"] == "84016"]["Price"].describe())

Sales rows reversed within 1 day: 1405
Revenue reversed: 404143.24


,Invoice,CancelInvoice,Description,Quantity,Revenue
13919,581483,C581484,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
8067,541431,C541433,MEDIUM CERAMIC TOP STORAGE JAR,74215,77183.60
10416,556444,C556448,PICNIC BASKET WICKER 60 PIECES,60,38970.00
12048,567423,C567527,PANTRY CHOPPING BOARD,756,3825.36
6367,529350,C529352,COLOUR GLASS. STAR T-LIGHT HOLDER,1152,3168.00
4014,515281,C515299,SET/4 WHITE RETRO STORAGE CUBES,85,2970.75
4016,515296,C515299,SET/4 WHITE RETRO STORAGE CUBES,85,2970.75
12333,569385,C569387,DOORMAT KEEP CALM AND COME IN,200,1416.00


count      62.000000
mean       53.197581
std       192.331744
min         0.350000
25%         0.420000
50%         0.420000
75%         0.420000
max      1157.150000
Name: Price, dtype: float64

In [10]:
# 1. Quick reversals and their cancellation rows
reversed_flag = sales.index.isin(reversed_idx)

cancellations["Customer ID"] = cancellations["Customer ID"].astype("Int64")
matched = (
    m[["CancelInvoice", "StockCode", "Customer ID"]]
    .drop_duplicates()
    .rename(columns={"CancelInvoice": "Invoice"})
)
canc_check = cancellations.merge(
    matched, on=["Invoice", "StockCode", "Customer ID"], how="left", indicator=True
)
canc_reversed = (canc_check["_merge"] == "both").to_numpy()

# 2. Price outliers relative to each product's typical price
med = sales.groupby("StockCode")["Price"].transform("median")
n = sales.groupby("StockCode")["Price"].transform("size")
price_outlier = ((sales["Price"] > 50 * med) & (n >= 5)).to_numpy()

print("Price outlier rows:", price_outlier.sum())
display(
    sales[price_outlier].nlargest(10, "Price")[
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
    ]
)

# 3. Final tables
reversed_orders = sales[reversed_flag].copy()
sales_final = sales[~reversed_flag & ~price_outlier].copy()
cancellations_final = cancellations[~canc_reversed].copy()

print("sales_final rows:", len(sales_final))
print("reversed_orders rows:", len(reversed_orders))
print("cancellations_final rows:", len(cancellations_final))
print("Revenue before:", round(sales["Revenue"].sum(), 2))
print("Revenue after:", round(sales_final["Revenue"].sum(), 2))

sales_final.to_csv("../data/sales_final.csv", index=False)
reversed_orders.to_csv("../data/reversed_orders.csv", index=False)
cancellations_final.to_csv("../data/cancellations_final.csv", index=False)

Price outlier rows: 14


,Invoice,StockCode,Description,Quantity,Price,Customer ID
192196,507637,84016,FLAG OF ST GEORGE CAR FLAG,1,1157.15,<NA>
136403,502451,84016,FLAG OF ST GEORGE CAR FLAG,1,867.79,<NA>
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,15098
748143,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,649.50,15098
180997,506571,84016,FLAG OF ST GEORGE CAR FLAG,1,408.40,<NA>
114155,500315,84016,FLAG OF ST GEORGE CAR FLAG,1,280.75,<NA>
248762,513439,84016,FLAG OF ST GEORGE CAR FLAG,1,272.27,<NA>
117876,500586,84016,FLAG OF ST GEORGE CAR FLAG,2,93.61,<NA>
136494,502469,84016,FLAG OF ST GEORGE CAR FLAG,1,93.61,<NA>
120056,500870,84033,FLAG OF ST GEORGE,1,85.10,<NA>


sales_final rows: 1001796
reversed_orders rows: 1405
cancellations_final rows: 16563
Revenue before: 19642692.15
Revenue after: 19234419.48


In [11]:

## Cleaning decisions

| Rule | Rows | Reason |
------------------------
| Exact duplicate rows dropped | 34,335 | Likely repeated entries (assumption: a genuine repeat purchase is also possible) |
| Cancellations (invoice starts with C) held separately | 17,914 | Needed for the returns analysis |
| Non-product stock codes excluded (postage, fees, adjustments) | several thousand | Would distort product revenue; meaning inferred from the codes |
| Zero or negative price, or non-cancellation negative quantity excluded | about 9,000 | Accounting adjustments, not sales |
| Orders reversed by an exact matching cancellation within 1 day excluded | 1,405 (£404,143) | Sales that never happened, including three orders worth about £285,000 |
| Prices over 50 times the product median excluded | 14 | Probable pricing or entry errors |
| Rows without a customer ID kept | 226,637 | Valid for sales analysis; dropped only for customer segmentation |

Limitations: reversals can only be matched when a customer ID exists, and two sales sharing one cancellation are both flagged.

SyntaxError: invalid character '£' (U+00A3) (698799297.py, line 9)